# Portfolio Optimization - Data Exploration (Indian Markets)

This notebook performs exploratory data analysis on Indian market instruments for portfolio optimization.

## Objectives
1. Download historical data for Indian ETFs and stocks (NIFTYBEES, BANKBEES, GOLDBEES, RELIANCE, INFY)
2. Compute daily log returns, monthly returns, volatility, and correlation matrix
3. Visualize volatility and correlation heatmap

## Indian Market Instruments Description
- **NIFTYBEES.NS**: Nippon India ETF Nifty BeES - Tracks Nifty 50 index (broad market exposure)
- **BANKBEES.NS**: Nippon India ETF Bank BeES - Banking sector ETF (sector-specific exposure)
- **GOLDBEES.NS**: Nippon India Gold BeES - Gold ETF (commodity exposure)
- **RELIANCE.NS**: Reliance Industries Ltd - Large-cap diversified conglomerate
- **INFY.NS**: Infosys Ltd - Large-cap IT services company (technology sector)

This diverse set provides exposure to broad market indices, specific sectors (banking, IT), commodities (gold), and individual large-cap stocks, offering good diversification for Indian market portfolio analysis.


In [ ]:
# Import required libraries
import sys
import os

# Add src directory to path to import our modules
sys.path.append(os.path.join(os.path.dirname(os.getcwd()), 'src'))
sys.path.append('../src')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

# Import our data ingestion module
from src.data_ingestion import (
    download_etf_data,
    compute_log_returns,
    compute_monthly_returns,
    compute_volatility,
    compute_correlation_matrix,
    get_etf_summary_stats
)

# Set plotting style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', None)


## 1. Download Historical Data

We'll download 5 years of historical data for our ETF portfolio. This provides a good balance between having enough data for statistical analysis while capturing recent market conditions.


In [ ]:
# Define Indian market tickers (NSE listed - .NS suffix required for yfinance)
indian_tickers = ['NIFTYBEES.NS', 'BANKBEES.NS', 'GOLDBEES.NS', 'RELIANCE.NS', 'INFY.NS']

# Download historical data (5 years)
print("Downloading historical data for Indian market instruments...")
prices = download_etf_data(indian_tickers, period='5y')

print(f"\nData shape: {prices.shape}")
print(f"Date range: {prices.index[0].date()} to {prices.index[-1].date()}")
print(f"\nFirst few rows:")
print(prices.head())
print(f"\nLast few rows:")
print(prices.tail())
print(f"\nBasic statistics:")
print(prices.describe())


## 2. Visualize Price Trends

Let's visualize how the prices of our ETFs have evolved over time. We'll normalize prices to start at 100 to compare relative performance.


In [ ]:
# Normalize prices to start at 100 for comparison
normalized_prices = (prices / prices.iloc[0]) * 100

# Create interactive plotly figure
fig = go.Figure()

for ticker in indian_tickers:
    fig.add_trace(go.Scatter(
        x=normalized_prices.index,
        y=normalized_prices[ticker],
        mode='lines',
        name=ticker,
        hovertemplate=f'{ticker}<br>Date: %{{x}}<br>Normalized Price: %{{y:.2f}}<extra></extra>'
    ))

fig.update_layout(
    title='Indian Market Instruments Price Performance (Normalized to 100)',
    xaxis_title='Date',
    yaxis_title='Normalized Price',
    hovermode='x unified',
    width=1000,
    height=600
)

fig.show()


## 3. Compute Daily Log Returns

Log returns are preferred in quantitative finance because:
1. They are symmetric (log returns of -50% and +50% don't cancel out)
2. They are time-additive (sum of daily log returns = total log return)
3. They approximate percentage returns for small changes
4. They are commonly used in portfolio optimization models


In [ ]:
# Compute daily log returns
daily_log_returns = compute_log_returns(prices)

# Remove first row (NaN)
daily_log_returns = daily_log_returns.dropna()

print(f"Daily log returns shape: {daily_log_returns.shape}")
print(f"\nFirst few rows:")
print(daily_log_returns.head())
print(f"\nSummary statistics:")
print(daily_log_returns.describe())


## 4. Compute Monthly Returns

Monthly returns are useful for longer-term analysis and can help reduce noise from daily fluctuations.


In [ ]:
# Compute monthly returns
monthly_returns = compute_monthly_returns(prices)

# Remove first row (NaN)
monthly_returns = monthly_returns.dropna()

print(f"Monthly returns shape: {monthly_returns.shape}")
print(f"\nFirst few rows:")
print(monthly_returns.head())
print(f"\nSummary statistics:")
print(monthly_returns.describe())


## 5. Compute Volatility

Volatility (standard deviation of returns) measures the risk or uncertainty of returns. Higher volatility indicates greater price swings.

We annualize volatility by multiplying daily volatility by √252 (trading days per year).


In [ ]:
# Compute annualized volatility
volatility = compute_volatility(daily_log_returns, annualized=True)

print("Annualized Volatility (Standard Deviation):")
print("=" * 50)
for ticker, vol in volatility.items():
    # Clean ticker name for display (remove .NS suffix)
    display_name = ticker.replace('.NS', '')
    print(f"{display_name}: {vol:.2%}")

# Create bar plot
fig, ax = plt.subplots(figsize=(10, 6))
# Clean ticker names for display
volatility_display = volatility.copy()
volatility_display.index = [ticker.replace('.NS', '') for ticker in volatility_display.index]
volatility_display.plot(kind='bar', ax=ax, color='steelblue', edgecolor='black')
ax.set_title('Annualized Volatility by Instrument', fontsize=14, fontweight='bold')
ax.set_xlabel('Instrument', fontsize=12)
ax.set_ylabel('Annualized Volatility', fontsize=12)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: '{:.1%}'.format(y)))
ax.grid(axis='y', alpha=0.3)
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()


## 6. Compute Correlation Matrix

Correlation measures how assets move together. Values range from -1 (perfect negative correlation) to +1 (perfect positive correlation).

- **High positive correlation** (>0.7): Assets tend to move together (e.g., SPY and QQQ)
- **Low correlation** (-0.3 to 0.3): Assets move independently
- **Negative correlation** (<-0.3): Assets tend to move in opposite directions (diversification benefit)

Understanding correlations is crucial for portfolio diversification.


In [ ]:
# Compute correlation matrix
correlation_matrix = compute_correlation_matrix(daily_log_returns)

print("Correlation Matrix:")
print("=" * 50)
print(correlation_matrix.round(3))

# Clean ticker names for display
correlation_matrix_display = correlation_matrix.copy()
correlation_matrix_display.columns = [ticker.replace('.NS', '') for ticker in correlation_matrix_display.columns]
correlation_matrix_display.index = [ticker.replace('.NS', '') for ticker in correlation_matrix_display.index]

# Create heatmap using seaborn
plt.figure(figsize=(10, 8))
mask = np.triu(np.ones_like(correlation_matrix_display, dtype=bool))  # Mask upper triangle
sns.heatmap(
    correlation_matrix_display,
    annot=True,
    fmt='.3f',
    cmap='RdYlBu_r',
    center=0,
    square=True,
    linewidths=1,
    cbar_kws={"shrink": 0.8},
    mask=mask,
    vmin=-1,
    vmax=1
)
plt.title('Indian Market Correlation Matrix (Daily Log Returns)', fontsize=14, fontweight='bold', pad=20)
plt.tight_layout()
plt.show()

# Create interactive plotly heatmap
fig = go.Figure(data=go.Heatmap(
    z=correlation_matrix_display.values,
    x=correlation_matrix_display.columns,
    y=correlation_matrix_display.index,
    colorscale='RdYlBu_r',
    zmid=0,
    text=correlation_matrix_display.values.round(3),
    texttemplate='%{text}',
    textfont={"size": 12},
    colorbar=dict(title="Correlation")
))

fig.update_layout(
    title='Indian Market Correlation Matrix (Interactive)',
    width=700,
    height=600
)

fig.show()


## 7. Return Distribution Analysis

Let's examine the distribution of returns to understand the risk characteristics of each ETF.


In [ ]:
# Plot return distributions
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.flatten()

for idx, ticker in enumerate(indian_tickers):
    ax = axes[idx]
    daily_log_returns[ticker].hist(bins=50, ax=ax, alpha=0.7, edgecolor='black')
    ax.axvline(daily_log_returns[ticker].mean(), color='red', linestyle='--', linewidth=2, label='Mean')
    display_name = ticker.replace('.NS', '')
    ax.set_title(f'{display_name} Daily Log Returns Distribution', fontweight='bold')
    ax.set_xlabel('Log Return')
    ax.set_ylabel('Frequency')
    ax.legend()
    ax.grid(alpha=0.3)

# Remove empty subplot
fig.delaxes(axes[5])

plt.tight_layout()
plt.show()


## 8. Summary Statistics Table

Let's create a comprehensive summary table with key metrics for each ETF.


In [ ]:
# Create summary statistics table
summary_stats = pd.DataFrame({
    'Mean Daily Return': daily_log_returns.mean(),
    'Annualized Mean Return': daily_log_returns.mean() * 252,
    'Volatility': volatility,
    'Sharpe Ratio (assume 0% risk-free rate)': (daily_log_returns.mean() * 252) / volatility,
    'Min Daily Return': daily_log_returns.min(),
    'Max Daily Return': daily_log_returns.max(),
    'Skewness': daily_log_returns.skew(),
    'Kurtosis': daily_log_returns.kurtosis()
})

print("Summary Statistics:")
print("=" * 80)
print(summary_stats.round(4))

# Display formatted table
summary_stats.style.format({
    'Mean Daily Return': '{:.4f}',
    'Annualized Mean Return': '{:.2%}',
    'Volatility': '{:.2%}',
    'Sharpe Ratio (assume 0% risk-free rate)': '{:.2f}',
    'Min Daily Return': '{:.4f}',
    'Max Daily Return': '{:.4f}',
    'Skewness': '{:.2f}',
    'Kurtosis': '{:.2f}'
})


## 9. Key Insights

### Observations:

1. **Volatility Ranking**: 
   - NIFTYBEES (broad market) typically shows moderate volatility
   - BANKBEES (banking sector) may show higher volatility due to sector-specific risks
   - GOLDBEES (gold) often shows lower volatility and acts as a hedge
   - Individual stocks (RELIANCE, INFY) may show varying volatility based on company-specific factors

2. **Correlation Patterns**:
   - NIFTYBEES and BANKBEES likely show high positive correlation (both equity-based)
   - GOLDBEES may have low or negative correlation with equities (diversification benefit)
   - RELIANCE and INFY (individual stocks) may show moderate correlation with NIFTYBEES
   - Sector-specific correlations (BANKBEES vs INFY) may vary based on market conditions

3. **Return Distributions**: Indian market returns typically show:
   - Negative skewness (more extreme negative returns)
   - High kurtosis (fat tails, more extreme events than normal distribution)
   - These characteristics are common in emerging markets like India

4. **Indian Market Characteristics**:
   - Higher volatility compared to developed markets
   - Strong sector-specific movements (banking, IT)
   - Gold often serves as a safe haven asset
   - Currency and macroeconomic factors influence returns

### Next Steps (Future Milestones):
- Portfolio optimization using mean-variance optimization
- Risk management with Value at Risk (VaR) and Conditional VaR
- Stress testing with historical scenarios (e.g., COVID-19, demonetization)
- Monte Carlo simulations for portfolio returns
- Consideration of Indian market-specific factors (currency, inflation, policy changes)

---

**Note**: This analysis provides the foundation for portfolio optimization in Indian markets. The correlation matrix and volatility estimates will be key inputs for optimization models in future milestones. Indian markets may exhibit different characteristics compared to developed markets, including higher volatility and different correlation patterns.
